In [ ]:
from rule_engine.builtins import Builtins

from control_limits import apply_rule, SieveData

data = SieveData(
    sieve_size=["3/4", "1/2", "3/8", "No.8", "No.4"],
    cum_passing_pct=[100.0, 60.0, 50.0, 90.0, 100.0],
    pc_retained=[0.0, 40.0, 50.0, 10.0, 0.0],
    cum_retained_pct=[100.0, 60.0, 50.0, 90.0, 100.0],
)
result1 = data.get_sieve_data('1/2')
result1

In [ ]:

rules = ['name == "1/2" and pc_retained > 30 and pc_retained < 60']
results = all([apply_rule(rule, result1) for rule in rules])

results

In [ ]:

# based on https://github.com/zeroSteiner/rule-engine/issues/70
from rule_engine import Context, Rule, resolve_attribute

CONDITION = (
    "is_valid_thing(things, 'thing1') ? things['thing1'] : is_valid_thing(things, 'thing2') ? things['thing2'] : null"
)


def is_valid_thing(things: dict, name: str) -> bool:
    return name in things and things[name]["isValid"]


class Container:
    def __init__(self, things: dict):
        self.things = things
        self.is_valid_thing = is_valid_thing


thing1 = {"name": "thing1", "isValid": True}
thing2 = {"name": "thing2", "isValid": False}
container = Container({"thing1": thing1, "thing2": thing2})
thing = Rule(CONDITION, context=Context(resolver=resolve_attribute)).evaluate(container)
print(thing)


In [ ]:
import rule_engine
# how to implement custom operators

from rule_engine import Context, Rule, resolve_attribute

rule = "is_larger(value, 10)"


def is_larger(thing: int, val: int) -> bool:
    return thing > val


class Val:
    def __init__(self, val: int):
        self.value = val
        self.is_larger = is_larger


thing1 = Val(val=2)
thing2 = Val(val=11)
r = Rule(rule, context=Context(resolver=resolve_attribute)).evaluate(thing1)
print(r)  # == False)
r = Rule(rule, context=Context(resolver=resolve_attribute)).evaluate(thing2)
print(r)  # == True)


In [ ]:
class CustomBuiltinsContext(rule_engine.Context):
    def __init__(self, *args, **kwargs):
        super(CustomBuiltinsContext, self).__init__(*args, **kwargs)
        self.builtins = Builtins.from_defaults(
            # expose the $version symbol
            {'version': rule_engine.__version__},
        )


rule_engine.Rule(
    '$version == "4.5.3"', context=CustomBuiltinsContext()
).matches({})


In [ ]:
# simple rule engine from the ground up
# https://dev.to/fractalis/how-to-write-a-basic-rule-engine-in-python-3eik

from functools import reduce
from typing import Callable, Any, List


class Fact:
    def __init__(self, **kwargs: Any):
        self.__dict__.update(kwargs)


class Action:

    def __init__(self, name: str, execution_function: Callable[[Fact], None]):
        self.name = name
        self.exec_func = execution_function

    def execute(self, fact: Fact) -> None:
        self.exec_func(fact)


class Condition:
    def __init__(self, name: str, evaluation_function: Callable[[Fact], bool]):
        self.name = name
        self.eval_func = evaluation_function

    def evaluate(self, fact: Fact) -> bool:
        return self.eval_func(fact)


class Rule:
    def __init__(self, condition: Condition, action: Action):
        self.conditions = [condition]
        self.actions = [action]

    def add_condition(self, condition: Condition) -> None:
        self.conditions.append(condition)

    def add_action(self, action: Action) -> None:
        self.actions.append(action)

    def evaluate(self, facts: List[Fact]) -> Any:
        def fact_generator(conditions: List[Condition], facts: List[Fact]):
            all_conditions_true = True
            for fact in facts:
                results = map(lambda condition: condition.eval_func(fact), conditions)
                all_conditions_true = reduce(lambda x, y: x and y, results)
                if all_conditions_true:
                    yield fact

        true_facts = list(fact_generator(self.conditions, facts))
        if len(true_facts) > 0:
            for fact in true_facts:
                for action in self.actions:
                    action.exec_func(fact)


age_cond = Condition(name="Age>=21", evaluation_function=lambda fact: fact.age >= 21)

occupation_cond = Condition(name="Occupation==Software Developer", evaluation_function=lambda fact: fact.occupation == "Software Developer")

print_action = Action(name="Print Fact", execution_function=lambda fact: print("Name: {} Age: {} Occupation: {}".format(fact.name, fact.age, fact.occupation)))

john = Fact(age=20, name="John Brown", occupation="Software Developer")
sarah = Fact(age=35, name="Sarah Purple", occupation="Data Engineer")
barry = Fact(age=27, name="Barry White", occupation="Software Developer")

rule = Rule(condition=age_cond, action=print_action)
rule.add_condition(occupation_cond)

rule.evaluate([john, sarah, barry])